In [1]:
import tarfile
import warnings
from glob import glob
from tqdm import tqdm

import anndata as ad
import matplotlib.pyplot as plt
import muon as mu
import pandas as pd
import numpy as np
import scanpy as sc
import scirpy as ir

sc.set_figure_params(figsize=(4, 4))
sc.settings.verbosity = 2 # verbosity: errors (0), warnings (1), info (2), hints (3)
pd.set_option('display.max_rows', 100)

# 读取TCR 

In [2]:
files = glob('/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/*')
files

['/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0003_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0002_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0003_WB.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0005_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0008_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0002_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0008_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0005_WB.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0008_WB.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0004_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0005_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_TCR_out/D0007_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_A

In [3]:
adatas_tcr = {}
for i in tqdm(files):
    sample_name = i.split('/')[-1].split('.')[0]
    airr_data = pd.read_csv(i)
    airr_data.columns=['cell_id', 'locus', 'v_call', 'd_call', 'j_call', 'c_call',
       'full_length', 'productive', 'junction_aa', 'junction', 'reads', 'umi_count','raw_clonotype_id']
    airr_data['cell_id'] = [sample_name + "_" + name for name in airr_data['cell_id']]
    adata = ir.io.read_airr(airr_data)
    adatas_tcr[sample_name] = adata

  0%|                                                                                           | 0/24 [00:00<?, ?it/s]/home/liyanguo/anaconda3/envs/R/lib/python3.12/site-packages/anndata/utils.py:349: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)
100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [00:17<00:00,  1.38it/s]


In [4]:
adata_tcr = ad.concat(adatas_tcr)

In [5]:
adata_tcr.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scTCR_ref.h5ad",compression="gzip")

# 读取BCR

In [6]:
files = glob('/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/*')
files

['/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0003_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0002_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0003_WB.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0005_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0008_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0002_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0008_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0005_WB.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0008_WB.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0004_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0005_Mix.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_Atlas_BCR_out/D0007_PBMC.csv',
 '/home/liyanguo/MyImmuCell/01_rawdata/Reference_A

In [7]:
adatas_bcr = {}
for i in tqdm(files):
    sample_name = i.split('/')[-1].split('.')[0]
    airr_data = pd.read_csv(i)
    airr_data.columns=['cell_id', 'locus', 'v_call', 'd_call', 'j_call', 'c_call',
       'full_length', 'productive', 'junction_aa', 'junction', 'reads', 'umi_count','raw_clonotype_id']
    airr_data['cell_id'] = [sample_name + "_" + name for name in airr_data['cell_id']]
    adata = ir.io.read_airr(airr_data)
    adatas_bcr[sample_name] = adata

100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [00:03<00:00,  7.99it/s]


In [8]:
adata_bcr = ad.concat(adatas_bcr)

In [9]:
adata_bcr.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scBCR_ref.h5ad",compression="gzip")

# 预处理VDJ数据

In [10]:
ir.pp.index_chains(adata_tcr)
ir.tl.chain_qc(adata_tcr)
ir.pp.index_chains(adata_bcr)
ir.tl.chain_qc(adata_bcr)

Filtering chains...
Indexing VJ chains...
Indexing VDJ chains...
build result array
Stored result in `adata.obs["receptor_type"]`.
Stored result in `adata.obs["receptor_subtype"]`.
Stored result in `adata.obs["chain_pairing"]`.
Filtering chains...
Indexing VJ chains...
Indexing VDJ chains...
build result array
Stored result in `adata.obs["receptor_type"]`.
Stored result in `adata.obs["receptor_subtype"]`.
Stored result in `adata.obs["chain_pairing"]`.


In [11]:
adata_tcr

AnnData object with n_obs × n_vars = 343000 × 0
    obs: 'receptor_type', 'receptor_subtype', 'chain_pairing'
    uns: 'chain_indices'
    obsm: 'airr', 'chain_indices'

In [12]:
data_tcr = ir.get.airr(adata_tcr, ["locus","v_call","j_call","junction_aa","junction","umi_count"],chain=('VJ_1', 'VDJ_1'))
data_tcr = adata_tcr.obs.join(data_tcr, how='left')

data_bcr = ir.get.airr(adata_bcr, ["locus","v_call","j_call","junction_aa","junction","umi_count"],chain=('VJ_1', 'VDJ_1'))
data_bcr = adata_bcr.obs.join(data_bcr, how='left')

In [13]:
data_tcr['receptor_type'].value_counts()

receptor_type
TCR    343000
Name: count, dtype: int64

In [14]:
data_bcr['receptor_type'].value_counts()

receptor_type
BCR    56152
Name: count, dtype: int64

In [15]:
data_tcr.to_csv("/home/liyanguo/MyImmuCell/02_Read_QC/scTCR_ref.csv")
data_bcr.to_csv("/home/liyanguo/MyImmuCell/02_Read_QC/scBCR_ref.csv")

In [16]:
data_tcr

,receptor_type,receptor_subtype,chain_pairing,VJ_1_locus,VJ_1_v_call,VJ_1_j_call,VJ_1_junction_aa,VJ_1_junction,VJ_1_umi_count,VDJ_1_locus,VDJ_1_v_call,VDJ_1_j_call,VDJ_1_junction_aa,VDJ_1_junction,VDJ_1_umi_count
cell_id,,,,,,,,,,,,,,,
D0003_Mix_AAACATCG_ACAGATTC_TGGCTTCA,TCR,TRA+TRB,single pair,TRA,TRAV34,TRAJ37,CGADKHGSGNTGKLIF,TGTGGAGCAGACAAACATGGCTCTGGCAACACAGGCAAACTAATCTTT,4,TRB,TRBV7-3,TRBJ2-1,CASSLIPSGRAVYNEQFF,TGTGCCAGCAGCTTAATACCTAGCGGGAGGGCCGTCTACAATGAGC...,73
D0003_Mix_AAACATCG_ATAGCGAC_GAGTTAGC,TCR,TRA+TRB,single pair,TRA,TRAV34,TRAJ37,CGADKHGSGNTGKLIF,TGTGGAGCAGACAAACATGGCTCTGGCAACACAGGCAAACTAATCTTT,9,TRB,TRBV7-3,TRBJ2-1,CASSLIPSGRAVYNEQFF,TGTGCCAGCAGCTTAATACCTAGCGGGAGGGCCGTCTACAATGAGC...,54
D0003_Mix_AAACATCG_CGCTGATC_CATCAAGT,TCR,TRA+TRB,single pair,TRA,TRAV34,TRAJ37,CGADKHGSGNTGKLIF,TGTGGAGCAGACAAACATGGCTCTGGCAACACAGGCAAACTAATCTTT,3,TRB,TRBV7-3,TRBJ2-1,CASSLIPSGRAVYNEQFF,TGTGCCAGCAGCTTAATACCTAGCGGGAGGGCCGTCTACAATGAGC...,117
D0003_Mix_AAACATCG_CGGATTGC_CAGCGTTA,TCR,TRA+TRB,single pair,TRA,TRAV34,TRAJ37,CGADKHGSGNTGKLIF,TGTGGAGCAGACAAACATGGCTCTGGCAACACAGGCAAACTAATCTTT,22,TRB,TRBV7-3,TRBJ2-1,CASSLIPSGRAVYNEQFF,TGTGCCAGCAGCTTAATACCTAGCGGGAGGGCCGTCTACAATGAGC...,41
D0003_Mix_AAACATCG_GAGTTAGC_AACCGAGA,TCR,TRA+TRB,single pair,TRA,TRAV34,TRAJ37,CGADKHGSGNTGKLIF,TGTGGAGCAGACAAACATGGCTCTGGCAACACAGGCAAACTAATCTTT,15,TRB,TRBV7-3,TRBJ2-1,CASSLIPSGRAVYNEQFF,TGTGCCAGCAGCTTAATACCTAGCGGGAGGGCCGTCTACAATGAGC...,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
D0006_WB_CAACCACA_CCTAATCC_TGGCTTCA,TCR,TRA+TRB,orphan VDJ,None,None,None,None,None,None,TRB,TRBV2,TRBJ2-7,CAGRGLAYEQYF,TGTGCCGGAAGGGGACTAGCCTACGAGCAGTACTTC,8
D0006_WB_AATGTTGC_TGAAGAGA_GAGCTGAA,TCR,TRA+TRB,orphan VDJ,None,None,None,None,None,None,TRB,TRBV16,TRBJ2-2,CAGAGTDTGELFF,TGTGCCGGGGCCGGGACCGACACCGGGGAGCTGTTTTTT,17
D0006_WB_CACCTTAC_GATGAATC_CAGCGTTA,TCR,TRA+TRB,orphan VDJ,None,None,None,None,None,None,TRB,TRBV7-3,TRBJ1-1,CADLGTGSWDTEAFF,TGTGCCGACTTGGGGACAGGGTCTTGGGACACTGAAGCTTTCTTT,54


In [17]:
data_bcr

,receptor_type,receptor_subtype,chain_pairing,VJ_1_locus,VJ_1_v_call,VJ_1_j_call,VJ_1_junction_aa,VJ_1_junction,VJ_1_umi_count,VDJ_1_locus,VDJ_1_v_call,VDJ_1_j_call,VDJ_1_junction_aa,VDJ_1_junction,VDJ_1_umi_count
cell_id,,,,,,,,,,,,,,,
D0003_Mix_AAACATCG_CGCTGATC_CGACTGGA,BCR,IGH+IGK,single pair,IGK,IGKV1-12,IGKJ1,CQQGDSFPPTF,TGTCAACAGGGTGACAGTTTCCCTCCGACATTC,14,IGH,IGHV4-30-2,IGHJ4,CARGEGFGQSVFDFW,TGTGCCAGAGGCGAAGGGTTCGGGCAGTCCGTCTTTGACTTCTGG,49
D0003_Mix_AACTCACC_AGTACAAG_ATAGCGAC,BCR,IGH+IGK,single pair,IGK,IGKV1-12,IGKJ1,CQQGDSFPPTF,TGTCAACAGGGTGACAGTTTCCCTCCGACATTC,33,IGH,IGHV4-30-2,IGHJ4,CARGEGFGQSVFDFW,TGTGCCAGAGGCGAAGGGTTCGGGCAGTCCGTCTTTGACTTCTGG,43
D0003_Mix_ACAGATTC_ATCCTGTA_AACGTGAT,BCR,IGH+IGK,single pair,IGK,IGKV1-12,IGKJ1,CQQGDSFPPTF,TGTCAACAGGGTGACAGTTTCCCTCCGACATTC,95,IGH,IGHV4-30-2,IGHJ4,CARGEGFGQSVFDFW,TGTGCCAGAGGCGAAGGGTTCGGGCAGTCCGTCTTTGACTTCTGG,205
D0003_Mix_ACATTGGC_TCCGTCTA_AACCGAGA,BCR,IGH+IGK,single pair,IGK,IGKV1-12,IGKJ1,CQQGDSFPPTF,TGTCAACAGGGTGACAGTTTCCCTCCGACATTC,9,IGH,IGHV4-30-2,IGHJ4,CARGEGFGQSVFDFW,TGTGCCAGAGGCGAAGGGTTCGGGCAGTCCGTCTTTGACTTCTGG,42
D0003_Mix_ACGCTCGA_GATAGACA_CAATGGAA,BCR,IGH+IGK,single pair,IGK,IGKV1-12,IGKJ1,CQQGDSFPPTF,TGTCAACAGGGTGACAGTTTCCCTCCGACATTC,23,IGH,IGHV4-30-2,IGHJ4,CARGEGFGQSVFDFW,TGTGCCAGAGGCGAAGGGTTCGGGCAGTCCGTCTTTGACTTCTGG,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
D0006_WB_GAATCTGA_AGAGTCAA_CCATCCTC,BCR,IGH+IGK,orphan VJ,IGK,IGKV2-29,IGKJ4,CLQGLHLPVTF,TGCTTGCAAGGTCTACACCTTCCGGTCACTTTC,24,None,None,None,None,None,None
D0006_WB_AACCGAGA_CACTTCGA_AACGTGAT,BCR,IGH+IGK,orphan VJ,IGK,IGKV1-6,IGKJ1,CLQDYNYPRTF,TGTCTACAAGATTACAATTACCCTCGGACGTTC,19,None,None,None,None,None,None
D0006_WB_ATCCTGTA_GTCTGTCA_ATTGGCTC,BCR,IGH+IGK,orphan VJ,IGK,IGKV3-15,IGKJ5,CHQYNNWPLTF,TGTCACCAGTATAATAACTGGCCCCTCACCTTC,12,None,None,None,None,None,None
